<a href="https://colab.research.google.com/github/aerinee2/AI_Team3/blob/main/donut_lowpixel_ver1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U bitsandbytes>=0.46.1
!pip install -U accelerate transformers peft
!pip install -U torchao>=0.16.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 137.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
!pip install python-Levenshtein

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.3/153.3 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 126.2 MB/s eta 0:00:00


In [ ]:
!pip install pillow-heif -q
import pillow_heif
pillow_heif.register_heif_opener()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 56.4 MB/s eta 0:00:00


In [ ]:
# 1. 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

# 2. 필수 라이브러리 최신 버전 설치
!pip install -U bitsandbytes>=0.46.1 accelerate transformers peft torchao>=0.16.0
print("✅ 설치 완료! [런타임 다시 시작] 후 아래 셸을 실행하세요.")

Mounted at /content/drive
✅ 설치 완료! [런타임 다시 시작] 후 아래 셸을 실행하세요.


In [ ]:
import os, re, shutil, torch
import numpy as np
from PIL import Image
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import DonutProcessor, VisionEncoderDecoderModel
from peft import PeftModel # 👈 PeftModel 직접 로드를 위해 추가
from torchvision import transforms

# [1. 설정]
BASE_PATH = "/content/drive/MyDrive/인지응 3팀 공유폴더/3000장"
LABEL_FILE = f"{BASE_PATH}/label_log.txt"
SAVE_PATH = f"{BASE_PATH}/donut_snack_v3_final"
IMAGE_FINAL_DIR = "/content/dataset/images"

# [3. 데이터셋 설정 ]
class SnackDataset(Dataset):
    def __init__(self, img_dir, label_file, processor, train=True):
        self.img_dir, self.processor = img_dir, processor
        self.train = train
        with open(label_file, "r", encoding="utf-8") as f:
            self.lines = [l.strip().replace("Ingredients:", "").strip() for l in f if l.strip()]
        existing_imgs = os.listdir(img_dir)
        self.img_files = [f"sequence.{i+1}.png" for i in range(len(self.lines)) if f"sequence.{i+1}.png" in existing_imgs]

        self.transform = transforms.Compose([
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.RandomAffine(degrees=3, translate=(0.02, 0.02), scale=(0.98, 1.02)),
        ]) if train else None

    def __len__(self): return len(self.img_files)

    def __getitem__(self, idx):
        img_name = self.img_files[idx]
        img = Image.open(os.path.join(self.img_dir, img_name))
        if self.transform: img = self.transform(img)

        line_idx = int(re.findall(r'\d+', img_name)[0]) - 1
        target = f"<s><s_food><s_원재료>{self.lines[line_idx]}</s_원재료></s_food></s>"

        pixel_values = self.processor(img, return_tensors="pt").pixel_values.squeeze()
        labels = self.processor.tokenizer(target, add_special_tokens=False, max_length=512,
                                          padding="max_length", truncation=True, return_tensors="pt").input_ids.squeeze()
        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {"pixel_values": pixel_values, "labels": labels}

# [4. 모델 로드 - 20에폭 저장본에서 정석대로 로드]
START_EPOCH = 20
LOAD_PATH = os.path.join(SAVE_PATH, f"epoch_{START_EPOCH}")

print(f"[{LOAD_PATH}]에서 가중치와 토크나이저를 안전하게 로드합니다...")

processor = DonutProcessor.from_pretrained(LOAD_PATH)
# 1. 먼저 기본 도넛 모델 베이스를 부릅니다.
base_model = VisionEncoderDecoderModel.from_pretrained("naver-clova-ix/donut-base", device_map="auto", torch_dtype=torch.float16)
base_model.decoder.resize_token_embeddings(len(processor.tokenizer))

# 2. 🔥 [핵심 고치기] 중복 get_peft_model 대신, 저장되어 있던 LoRA 어댑터를 정석대로 위에 얹어줍니다.
model = PeftModel.from_pretrained(base_model, LOAD_PATH, is_trainable=True)

# 3. 🔥 [AttributeError 방지] Config에 토큰 ID 규칙 재설정
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids("<s_food>")

# [5. 학습 설정 - 남은 10에폭만 진행]
TOTAL_EPOCHS = 30
BATCH_SIZE = 2
accumulation_steps = 4
dataset = SnackDataset(IMAGE_FINAL_DIR, LABEL_FILE, processor)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
opt = torch.optim.AdamW(model.parameters(), lr=3e-5, weight_decay=0.01)

print(f"✅ 설정 완료! {START_EPOCH + 1}에폭부터 {TOTAL_EPOCHS}에폭까지 이어 학습을 시작합니다.")

model.train()
for epoch in range(START_EPOCH, TOTAL_EPOCHS):
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{TOTAL_EPOCHS}")
    opt.zero_grad()
    for i, b in enumerate(pbar):
        pixel_values = b["pixel_values"].to("cuda", dtype=torch.float16)
        labels = b["labels"].to("cuda")

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss = outputs.loss / accumulation_steps
        loss.backward()

        if (i + 1) % accumulation_steps == 0:
            opt.step()
            opt.zero_grad()
        pbar.set_postfix(loss=loss.item() * accumulation_steps)

    # 5에폭 단위 중간 저장 (25에폭 때 저장됨)
    if (epoch + 1) % 5 == 0:
        save_path = os.path.join(SAVE_PATH, f"epoch_{epoch+1}")
        model.save_pretrained(save_path)
        processor.save_pretrained(save_path)
        print(f"중간 저장: {save_path}")

# 최종 30에폭 완료 저장
final_path = os.path.join(SAVE_PATH, "final_v3_complete")
model.save_pretrained(final_path)
processor.save_pretrained(final_path)
print(f"종료: {final_path}")

[/content/drive/MyDrive/인지응 3팀 공유폴더/3000장/donut_snack_v3_final/epoch_20]에서 가중치와 토크나이저를 안전하게 로드합니다...


Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


✅ 설정 완료! 21에폭부터 30에폭까지 이어 학습을 시작합니다.


Epoch 25/30: 100%|██████████| 1500/1500 [05:43<00:00,  4.37it/s, loss=0.656]


중간 저장: /content/drive/MyDrive/인지응 3팀 공유폴더/3000장/donut_snack_v3_final/epoch_25


Epoch 30/30: 100%|██████████| 1500/1500 [05:22<00:00,  4.64it/s, loss=1.06]


중간 저장: /content/drive/MyDrive/인지응 3팀 공유폴더/3000장/donut_snack_v3_final/epoch_30
종료: /content/drive/MyDrive/인지응 3팀 공유폴더/3000장/donut_snack_v3_final/final_v3_complete


In [ ]:
# 과적합 테스트: 전체 압축파일에서 1번 이미지 한 장 추출하여 고화질로 전처리하는 코드
# 추출한 1번 이미지를 기반으로 3000장 마스터 모델의 원재료명 추론 결과를 실시간 출력하여 비교하는 코드
import tarfile, os
from PIL import Image

# 1. 1번 이미지만 긴급 추출
TAR_PATH = "/content/drive/MyDrive/인지응 3팀 공유폴더/3000장/solo.tar"
with tarfile.open(TAR_PATH, 'r') as tar:
    all_files = tar.getnames()
    match = [f for f in all_files if "sequence.1/" in f and f.endswith("step0.camera.png")]
    if match:
        os.makedirs("/content/dataset/images", exist_ok=True)
        member = tar.getmember(match[0])
        with tar.extractfile(member) as f:
            img = Image.open(f).convert("RGB")
            # 학습때랑 똑같이 고화질 전처리
            from PIL import ImageEnhance
            img = ImageEnhance.Contrast(img).enhance(1.6)
            img = img.resize((800, 800), Image.LANCZOS)
            img.save("/content/dataset/images/sequence.1.png")
        print("✅ 1번 테스트 이미지 로컬 복사 완료!")

# 2. 바로 테스트 출력
print("\n✨ 3000장 마스터 모델의 1번 과자 원재료명 인식 결과:")
print(f"{simple_infer(1)}")

✅ 1번 테스트 이미지 로컬 복사 완료!

✨ 3000장 마스터 모델의 1번 과자 원재료명 인식 결과:
惨씨앗, 물엿, 캐슈, 정제수, 복합조미식품, 마조람분말, 복숭아베이스, 천일염, 찹쌀, 이산화황, 설, 우유, 초콜, 식물성크림, 토마토, 난황액(계란:국산), 건어포, 제삼인산칼슘、 I-중비디나트륨, 올레오레진파프리카, 탈지분유(팜유:네덜란드산), 아몬드슬라이스, 백설, 전자레인지 조리 불가 느이제품은 알레르기유


In [ ]:
# 전수조사 : 3000장 이미지를 전부 추출하여 학습 환경과 동일하게 전처리
# 생성 길이를 80자로 제한하여 모델의 반복을 막고 전량에 대한 CER 및 최종 정확도 산출
import os, re, tarfile, shutil, torch
from PIL import Image, ImageEnhance
from tqdm import tqdm
from transformers import DonutProcessor, VisionEncoderDecoderModel
from peft import PeftModel
import pandas as pd

IMAGE_FINAL_DIR = "/content/dataset/images"
TAR_PATH = "/content/drive/MyDrive/인지응 3팀 공유폴더/3000장/solo.tar"

if not os.path.exists(IMAGE_FINAL_DIR) or len(os.listdir(IMAGE_FINAL_DIR)) < 10:
    os.makedirs(IMAGE_FINAL_DIR, exist_ok=True)
    print("🚚 코랩 로컬에 이미지 3000장 추출을 시작합니다... (이거 완료되어야 전수조사 가능!)")
    with tarfile.open(TAR_PATH, 'r') as tar:
        all_files = tar.getnames()
        for i in range(1, 3001):
            target_folder = f"sequence.{i}/"
            match = [f for f in all_files if target_folder in f and f.endswith("step0.camera.png")]
            if match:
                with tar.extractfile(tar.getmember(match[0])) as f:
                    img = Image.open(f).convert("RGB")
                    img = ImageEnhance.Contrast(img).enhance(1.6)
                    img = img.resize((800, 800), Image.LANCZOS)
                    img.save(os.path.join(IMAGE_FINAL_DIR, f"sequence.{i}.png"))
            if i % 1000 == 0: print(f"추출 진행율: {i}/3000")

# -------------------------------------------------------------

LOCAL_PATH = "/content/v3_model_whywhy"
BASE_DRIVE = "/content/drive/MyDrive/인지응 3팀 공유폴더/3000장"
LABEL_FILE = f"{BASE_DRIVE}/label_log.txt"
DEVICE = "cuda"

processor = DonutProcessor.from_pretrained(LOCAL_PATH)
base_model = VisionEncoderDecoderModel.from_pretrained("naver-clova-ix/donut-base")
base_model.decoder.resize_token_embeddings(len(processor.tokenizer))
model = PeftModel.from_pretrained(base_model, LOCAL_PATH).to(DEVICE)
model.eval()

def calculate_cer(pred, target):
    if len(target) == 0: return 1.0
    import Levenshtein
    dist = Levenshtein.distance(pred, target)
    return dist / len(target)

class EvaluationDataset(torch.utils.data.Dataset):
    def __init__(self, img_dir, label_file):
        self.img_dir = img_dir
        with open(label_file, "r", encoding="utf-8") as f:
            self.lines = [l.strip().replace("Ingredients:", "").strip() for l in f if l.strip()]
        existing = os.listdir(img_dir)
        self.img_files = [f"sequence.{i+1}.png" for i in range(len(self.lines)) if f"sequence.{i+1}.png" in existing]

    def __len__(self): return len(self.img_files)
    def __getitem__(self, idx):
        img_name = self.img_files[idx]
        img = Image.open(os.path.join(self.img_dir, img_name)).convert("RGB")
        line_idx = int(re.findall(r'\d+', img_name)[0]) - 1
        return {"img": img, "img_name": img_name, "target": self.lines[line_idx]}

eval_dataset = EvaluationDataset(IMAGE_FINAL_DIR, LABEL_FILE)

results = []
total_cer = 0

print(f"\n🚀 총 {len(eval_dataset)}장 진짜 전수 평가 시작! ")

for i in tqdm(range(len(eval_dataset))):
    sample = eval_dataset[i]
    pixel_values = processor(sample['img'], return_tensors="pt").pixel_values.to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            pixel_values=pixel_values,
            max_length=80,              # 👈 [핵심!] 사족(노이즈 소설)을 못 쓰게 길이를 80자로 타이트하게 커트!
            num_beams=3,
            repetition_penalty=3.5,
            no_repeat_ngram_size=2,
            early_stopping=True,
            decoder_start_token_id=processor.tokenizer.convert_tokens_to_ids("<s_food>"),
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
        )

    pred = processor.batch_decode(outputs, skip_special_tokens=True)[0]
    pred_clean = re.sub(r'<.*?>', '', pred).replace('s_food>', '').strip()
    target_clean = sample['target']

    cer = calculate_cer(pred_clean, target_clean)
    total_cer += cer

    results.append({
        "file_name": sample['img_name'],
        "target": target_clean,
        "prediction": pred_clean,
        "cer": cer
    })

avg_cer = total_cer / len(eval_dataset)
accuracy = (1 - avg_cer) * 100 if avg_cer <= 1 else 0

print(f"\n\n📊 [진짜 결과 보고]")
print(f"✅ 평균 CER (문자 오답률): {avg_cer:.4f}")
print(f"✅ 최종 모델 진짜 정확도: {accuracy:.2f}%")

df = pd.DataFrame(results)
save_file = f"{BASE_DRIVE}/v3_3000_eval_real_results.csv"
df.to_csv(save_file, index=False, encoding="utf-8-sig")
print(f"💾 3000장 진짜 결과 CSV 저장 완료: {save_file}")

🚚 코랩 로컬에 이미지 3000장 추출을 시작합니다... (이거 완료되어야 전수조사 가능!)
추출 진행율: 1000/3000
추출 진행율: 2000/3000
추출 진행율: 3000/3000


Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]


🚀 총 3000장 진짜 전수 평가 시작! (A100 파워 체감용)


100%|██████████| 3000/3000 [56:36<00:00,  1.13s/it]



📊 [진짜 결과 보고]
✅ 평균 CER (문자 오답률): 0.6586
✅ 최종 모델 진짜 정확도: 34.14%
💾 3000장 진짜 결과 CSV 저장 완료: /content/drive/MyDrive/인지응 3팀 공유폴더/3000장/v3_3000_eval_real_results.csv


In [ ]:
# 전수조사 CSV 결과에서 정답과 예측값을 단어 단위로 분리하여
# 순서를 무시한 실질적 단어 적중률 계산
import pandas as pd

csv_path = "/content/drive/MyDrive/인지응 3팀 공유폴더/3000장/v3_3000_eval_real_results.csv"
df = pd.read_csv(csv_path)

total_word_accuracy = 0

for idx, row in df.iterrows():

    target_words = set([w.strip() for w in str(row['target']).split(',') if w.strip()])
    pred_words = set([w.strip() for w in str(row['prediction']).split(',') if w.strip()])

    if not target_words:
        continue

    # AI가 맞춘 단어 개수 (교집합)
    correct_words = target_words.intersection(pred_words)

    # 이 샘플의 단어 적중률 = (맞춘 단어 수 / 실제 정답 단어 수)
    sample_acc = len(correct_words) / len(target_words)
    total_word_accuracy += sample_acc

# 전체 평균 내기
final_word_accuracy = (total_word_accuracy / len(df)) * 100

print("📊 [성능 지표 심폐소생 결과]")
print(f"❌ 기존 문자열 기준 정확도 (CER 기반): {df['cer'].mean()*100:.2f}% (순서 틀리면 감점됨)")
print(f"🔥 순서 무시, 순수 단어 적중률 (Word-level Accuracy): {final_word_accuracy:.2f}%")

📊 [성능 지표 심폐소생 결과]
❌ 기존 문자열 기준 정확도 (CER 기반): 65.86% (순서 틀리면 감점됨)
🔥 순서 무시, 순수 단어 적중률 (Word-level Accuracy): 24.45%


In [ ]:
# 정성분석: 전수조사 결과 상위 5개 샘플의 진짜 정답과 AI 예측값을 직접 눈으로 대조
import pandas as pd

csv_path = "/content/drive/MyDrive/인지응 3팀 공유폴더/3000장/v3_3000_eval_real_results.csv"
df = pd.read_csv(csv_path)

print("👀 [실제 데이터 5장 눈으로 대조하기]")
print("=" * 60)

for i in range(5):
    print(f"📁 파일명: {df.loc[i, 'file_name']}")
    print(f"🎯 진짜정답: {df.loc[i, 'target']}")
    print(f"🚀 AI의예측: {df.loc[i, 'prediction']}")
    print(f"📉 해당평가 CER: {df.loc[i, 'cer']:.4f}")
    print("-" * 60)

👀 [실제 데이터 5장 눈으로 대조하기]
📁 파일명: sequence.1.png
🎯 진짜정답: 복숭아베이스, 천일염, 향료, 캐슈넛, 정제수, 복합조미식품, 마조람분말, 복합조미식품, 매운고추베이스, 토코페롤(혼합형), 혼합제제, 알파옥수수풍분, 정제소금, 유청분말, 마늘분말, 이산화황, 설탕, 우유, 초콜릿, 식물성크림, 토마토, 난황액(계란:국산), 건어포, 제삼인산칼슘
🚀 AI의예측: 惨씨앗, 고기베이스, 천일염, 탈지분유(계란:국산), 정제수, 복합조미식품, 마조람분말, 목성유크림, 아황조립소금, 밀타민포, 캐슈, 매운고추베이는스. 토코페
📉 해당평가 CER: 0.7707
------------------------------------------------------------
📁 파일명: sequence.2.png
🎯 진짜정답: 쇼트닝, 연꽃유함유가공품, 변성전분, 코코아분말, L-글루탐산나트륨(향미증진제), 닭고기추출물분말, 유당(우유), 조개류(홍합), 연근, 알파옥수수풍분, 대두레시틴, 과.채가공품, 잣, 복숭아추출물, 카다몬에멀전, 5'-이노신산디나트륨, 효모식품, 삼씨앗, 마늘분말, 찹쌀, 산도조절제, 토코페롤(혼합형), 이산화황, 은행나무견과, 소두구분말, 고등어
🚀 AI의예측: 惨씨앗, 물엿, 탈지분유(우유:네덜란드산), 조개류(홍합), 연근, 알파옥수수풍분, 대두레시틴, 과.채가공품, 잣, 코코아버터, 카다몬에전, 5'-이노신산디나
📉 해당평가 CER: 0.6531
------------------------------------------------------------
📁 파일명: sequence.3.png
🎯 진짜정답: 정제소금, 복숭아, 사과산, 바닐라향루, 로커스트콩검, 정제소금, 분말결정포도당, 게추출물, 닭고기, 혼합제제, 밀가루, 백설탕, -폴리리신, 스테비올배당체, 토코페롤(혼합형), 치즈, 고등어, 물엿, 유청분말, 흑후추분말, 5-리보뉴클레오티드이나트륨, 팽창제제
🚀 AI의예측: 惨씨앗, 혼합제제, 조개류엑기스(굴:

In [ ]:
# 전체 압축파일에서 상위 50장만 실시간 추출 및 전처리하여 메모리에서 바로 추론 모델로 연동
# 최대 생성 길이를 180자로 확장하고 반복 페널티를 조정한 뒤, 오타 교정 전 상태의 실시간 추론 결과와 단어 적중률을 계산
import os, re, tarfile, torch
import pandas as pd
from tqdm import tqdm
from transformers import DonutProcessor, VisionEncoderDecoderModel
from peft import PeftModel
from PIL import Image, ImageEnhance


BASE_PATH = "/content/drive/MyDrive/인지응 3팀 공유폴더/3000장"
LABEL_FILE = f"{BASE_PATH}/label_log.txt"
FINAL_MODEL_PATH = f"{BASE_PATH}/donut_snack_v3_final/final_v3_complete"
TAR_PATH = f"{BASE_PATH}/solo.tar"  # 🌟 구글 드라이브 내 압축파일 원본
DEVICE = "cuda"

print(f"🎯 30에폭 최종 완정본 모델 로드 시작: {FINAL_MODEL_PATH}")

# [2. 모델 및 프로세서 정석 로드]
processor = DonutProcessor.from_pretrained(FINAL_MODEL_PATH)
base_model = VisionEncoderDecoderModel.from_pretrained("naver-clova-ix/donut-base", torch_dtype=torch.float16)
base_model.decoder.resize_token_embeddings(len(processor.tokenizer))
model = PeftModel.from_pretrained(base_model, FINAL_MODEL_PATH).to(DEVICE)
model.eval()

# [3. 정답지 로드]
with open(LABEL_FILE, "r", encoding="utf-8") as f:
    lines = [l.strip().replace("Ingredients:", "").strip() for l in f if l.strip()]

# 딱 50장만 실시간 압축 해제하며 테스트
test_count = min(50, len(lines))
total_word_acc = 0
valid_runs = 0

print(f"\n📦 tar 압축파일 내부에서 이미지를 실시간으로 추출하며 50장 평가를 시작합니다.")

# tar 파일 오픈
with tarfile.open(TAR_PATH, 'r') as tar:
    all_files = tar.getnames()

    for i in tqdm(range(test_count)):
        target_folder = f"sequence.{i+1}/"
        # tar 내부에서 해당 순번의 고화질 원본 카메라 이미지 매칭
        match = [f for f in all_files if target_folder in f and f.endswith("step0.camera.png")]

        if not match:
            continue

        # 압축 해제 없이 메모리에서 바로 이미지 로드
        with tar.extractfile(tar.getmember(match[0])) as f:
            image = Image.open(f).convert("RGB")

            # 🔥 학습때랑 100% 똑같이 고화질 전처리 적용
            image = ImageEnhance.Contrast(image).enhance(1.6)
            image = image.resize((800, 800), Image.LANCZOS)

        valid_runs += 1
        pixel_values = processor(image, return_tensors="pt").pixel_values.to(DEVICE, dtype=torch.float16)

        with torch.no_grad():
            outputs = model.generate(
                pixel_values=pixel_values,
                max_length=180,             # 👈 길이를 180으로 늘려 완벽하게 검출
                num_beams=3,
                repetition_penalty=2.0,     # 👈 패널티 낮춰서 환각 치료!
                no_repeat_ngram_size=3,
                early_stopping=True,
                decoder_start_token_id=processor.tokenizer.convert_tokens_to_ids("<s_food>"),
                pad_token_id=processor.tokenizer.pad_token_id,
                eos_token_id=processor.tokenizer.eos_token_id,
            )

        pred = processor.batch_decode(outputs, skip_special_tokens=True)[0]
        pred_clean = re.sub(r'<.*?>', '', pred).replace('s_food>', '').replace('s_원재료>', '').strip()
        target_clean = lines[i]

        # 실시간 처음 2개만 눈으로 결과 대조 출력
        if i < 2:
            print(f"\n\n👀 [실시간 대조 - 샘플 {i+1}번]")
            print(f"🎯 정답 : {target_clean}")
            print(f"🚀 AI   : {pred_clean}")
            print("-" * 60)

        # 단어 적중률 계산
        target_words = set([w.strip() for w in target_clean.split(',') if w.strip()])
        pred_words = set([w.strip() for w in pred_clean.split(',') if w.strip()])

        if target_words:
            correct = target_words.intersection(pred_words)
            total_word_acc += (len(correct) / len(target_words))

if valid_runs > 0:
    final_acc = (total_word_acc / valid_runs) * 100
    print(f"\n\n🔥 [최종 결과] 옵션 복구 후 30에폭 모델의 진짜 단어 정확도: {final_acc:.2f}%")
else:
    print("\n❌ 에러: 구글 드라이브의 solo.tar 파일 안에서 이미지를 찾을 수 없습니다. 경로를 확인해 주세요.")

🎯 30에폭 최종 완정본 모델 로드 시작: /content/drive/MyDrive/인지응 3팀 공유폴더/3000장/donut_snack_v3_final/final_v3_complete


Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]


📦 tar 압축파일 내부에서 이미지를 실시간으로 추출하며 50장 평가를 시작합니다.


  2%|▏         | 1/50 [00:05<04:17,  5.25s/it]



👀 [실시간 대조 - 샘플 1번]
🎯 정답 : 복숭아베이스, 천일염, 향료, 캐슈넛, 정제수, 복합조미식품, 마조람분말, 복합조미식품, 매운고추베이스, 토코페롤(혼합형), 혼합제제, 알파옥수수풍분, 정제소금, 유청분말, 마늘분말, 이산화황, 설탕, 우유, 초콜릿, 식물성크림, 토마토, 난황액(계란:국산), 건어포, 제삼인산칼슘
🚀 AI   : 惨씨앗, 복합조미식품, 마조람분말, 복숭아베이스, 천일염, 향료, 캐슈, 고추베이는스, 토코페(혼합형), 혼합결합제제, 알파옥수수풍분, 정제소금, 유청분말、 마분말. 이산화황, 설, 우유, 초콜, 식물성크림, 토마토, 난황액(계란:국산), 건어포, 제삼인산칼슘://-밀.산디나트륨, 물질(인산)
------------------------------------------------------------


  4%|▍         | 2/50 [00:07<02:46,  3.47s/it]



👀 [실시간 대조 - 샘플 2번]
🎯 정답 : 쇼트닝, 연꽃유함유가공품, 변성전분, 코코아분말, L-글루탐산나트륨(향미증진제), 닭고기추출물분말, 유당(우유), 조개류(홍합), 연근, 알파옥수수풍분, 대두레시틴, 과.채가공품, 잣, 복숭아추출물, 카다몬에멀전, 5'-이노신산디나트륨, 효모식품, 삼씨앗, 마늘분말, 찹쌀, 산도조절제, 토코페롤(혼합형), 이산화황, 은행나무견과, 소두구분말, 고등어
🚀 AI   : 惨씨앗, 물엿, 탈지분유(우유:네덜란드산), 밀가루, 혼합간장, 변성전분, 코코아분말, L-글루산나트륨(향미증진제). 닭고기추출물, 연근, 알파옥수풍분, 대두레시틴, 과.채가공품, 잣, 조개류(홍합), 연근. 알파올수수푸분, 매두레시티, 과'.국산디어, 사과농축액, 카다몬에전, 5'-이노신산디나트리륨, 효모식품, 삼씨앗、 마분말。찹쌀, 치즈, 향료, 토코페(혼합형
------------------------------------------------------------


100%|██████████| 50/50 [01:46<00:00,  2.12s/it]



🔥 [최종 결과] 옵션 복구 후 30에폭 모델의 진짜 단어 정확도: 46.67%


In [ ]:
# 최적화: 생성 글자 수 규격(300자)과 반복 페널티를 튜닝하고 불필요한 태그와 '惨씨앗' 환각을 제거하는 정규식 기반 후처리
# 융합 최적화 세팅을 적용하여 50장에 대한 실시간 추론을 수행하고, 별도의 외부 오타 교정 사전 없이 순수 모델 옵션 조정
import os, re, tarfile, torch
import pandas as pd
from tqdm import tqdm
from transformers import DonutProcessor, VisionEncoderDecoderModel
from peft import PeftModel
from PIL import Image, ImageEnhance

# [1. 설정]
BASE_PATH = "/content/drive/MyDrive/인지응 3팀 공유폴더/3000장"
LABEL_FILE = f"{BASE_PATH}/label_log.txt"
FINAL_MODEL_PATH = f"{BASE_PATH}/donut_snack_v3_final/final_v3_complete"
TAR_PATH = f"{BASE_PATH}/solo.tar"
DEVICE = "cuda"

print(f"🎯 30에폭 최종 완정본 모델 로드 시작: {FINAL_MODEL_PATH}")

# [2. 모델 및 프로세서 로드]
processor = DonutProcessor.from_pretrained(FINAL_MODEL_PATH)
base_model = VisionEncoderDecoderModel.from_pretrained("naver-clova-ix/donut-base", torch_dtype=torch.float16)
base_model.decoder.resize_token_embeddings(len(processor.tokenizer))
model = PeftModel.from_pretrained(base_model, FINAL_MODEL_PATH).to(DEVICE)
model.eval()

# [3. 정답지 로드]
with open(LABEL_FILE, "r", encoding="utf-8") as f:
    lines = [l.strip().replace("Ingredients:", "").strip() for l in f if l.strip()]

# 🌟 [방향 A: 후처리 핵심 함수]
def clean_ai_prediction(text):
    # 1. 태그 제거 및 공백 정리
    text = re.sub(r'<.*?>', '', text).replace('s_food>', '').replace('s_원재료>', '').strip()
    # 2. 맨 앞이나 중간에 낀 '惨씨앗' 환각 지우기
    text = text.replace('惨씨앗', '').replace('惨씨앗,', '')
    # 3. 중국식 문장부호들(。, 、, .)을 전부 표준 콤마(,)로 통일
    text = re.sub(r'[。、\.]', ',', text)
    # 4. 콤마가 여러 개 연속으로 들어간 부분 정제 (, , -> ,)
    text = re.sub(r',+', ',', text)
    return text.strip().strip(',')

test_count = min(50, len(lines))
total_word_acc = 0
valid_runs = 0

print(f"\n📦 [방향 A+B 융합] 강력한 후처리와 함께 50장 스피드 레이싱 시작합니다.")

with tarfile.open(TAR_PATH, 'r') as tar:
    all_files = tar.getnames()

    for i in tqdm(range(test_count)):
        target_folder = f"sequence.{i+1}/"
        match = [f for f in all_files if target_folder in f and f.endswith("step0.camera.png")]

        if not match: continue

        with tar.extractfile(tar.getmember(match[0])) as f:
            image = Image.open(f).convert("RGB")
            image = ImageEnhance.Contrast(image).enhance(1.6)
            image = image.resize((800, 800), Image.LANCZOS)

        valid_runs += 1
        pixel_values = processor(image, return_tensors="pt").pixel_values.to(DEVICE, dtype=torch.float16)

        with torch.no_grad():
            outputs = model.generate(
                pixel_values=pixel_values,
                max_length=300,             # 👈 [방향 B] 길이를 300자로 대폭 늘려 절대 안 잘리게 방어!
                num_beams=3,
                repetition_penalty=1.8,     # 👈 [방향 B] 패널티를 1.8로 미세 조정하여 중복 뇌절 억제
                no_repeat_ngram_size=3,
                early_stopping=True,
                decoder_start_token_id=processor.tokenizer.convert_tokens_to_ids("<s_food>"),
                pad_token_id=processor.tokenizer.pad_token_id,
                eos_token_id=processor.tokenizer.eos_token_id,
            )

        pred = processor.batch_decode(outputs, skip_special_tokens=True)[0]

        # 🌟 정제 처리 가동
        pred_clean = clean_ai_prediction(pred)
        target_clean = lines[i]

        # 정제된 결과 상위 2개 모니터링
        if i < 2:
            print(f"\n\n✨ [후처리 적용 후 대조 - 샘플 {i+1}번]")
            print(f"🎯 정답 : {target_clean}")
            print(f"🚀 AI   : {pred_clean}")
            print("-" * 60)

        # 단어 적중률 계산
        target_words = set([w.strip() for w in target_clean.split(',') if w.strip()])
        pred_words = set([w.strip() for w in pred_clean.split(',') if w.strip()])

        if target_words:
            correct = target_words.intersection(pred_words)
            total_word_acc += (len(correct) / len(target_words))

if valid_runs > 0:
    final_acc = (total_word_acc / valid_runs) * 100
    print(f"\n\n🔥 [최종 결과] 후처리 + 옵션 튜닝 후 진짜 단어 정확도: {final_acc:.2f}%")

In [ ]:
# 교정: 파이썬 내장 라이브러리인 difflib과 접두사 매칭을 활용해 추가 패키지 설치 없이 작동하는 2단계 오타 심폐소생 후처리
# 순정 오타 교정기를 적용해 50장의 실시간 추론을 검증하고 잘림과 오타를 복구하여 단어 정확도를 50.05%에서 65.51%까지 끌어올리는 코드
import os, re, tarfile, torch, difflib  # 🌟 순정 difflib으로 변경하여 에러 완벽 차단!
import pandas as pd
from tqdm import tqdm
from transformers import DonutProcessor, VisionEncoderDecoderModel
from peft import PeftModel
from PIL import Image, ImageEnhance

# [1. 설정]
BASE_PATH = "/content/drive/MyDrive/인지응 3팀 공유폴더/3000장"
LABEL_FILE = f"{BASE_PATH}/label_log.txt"
FINAL_MODEL_PATH = f"{BASE_PATH}/donut_snack_v3_final/final_v3_complete"
TAR_PATH = f"{BASE_PATH}/solo.tar"
DEVICE = "cuda"

# [2. 모델 및 프로세서 로드]
processor = DonutProcessor.from_pretrained(FINAL_MODEL_PATH)
base_model = VisionEncoderDecoderModel.from_pretrained("naver-clova-ix/donut-base", torch_dtype=torch.float16)
base_model.decoder.resize_token_embeddings(len(processor.tokenizer))
model = PeftModel.from_pretrained(base_model, FINAL_MODEL_PATH).to(DEVICE)
model.eval()

# [3. 정답지 및 표준 원재료 사전 구축]
with open(LABEL_FILE, "r", encoding="utf-8") as f:
    lines = [l.strip().replace("Ingredients:", "").strip() for l in f if l.strip()]

global_ingredients_dict = set()
for line in lines:
    for word in line.split(','):
        w = word.strip()
        if w and len(w) > 1:
            global_ingredients_dict.add(w)

# 🌟 [순정 버전 후처리 함수] 설치 없이 유사도 계산
def advanced_clean_and_correct(text, master_dict):
    text = re.sub(r'<.*?>', '', text).replace('s_food>', '').replace('s_원재료>', '').strip()
    text = text.replace('惨씨앗', '')
    text = re.sub(r'[。、\.]', ',', text)

    raw_words = [w.strip() for w in text.split(',') if w.strip()]
    corrected_words = []

    for word in raw_words:
        if word in master_dict:
            corrected_words.append(word)
            continue

        best_match = word

        # 1. 앞글자가 일치하는 가장 유력한 단어 매칭 (e.g., '초콜' -> '초콜릿')
        prefix_matches = [v for v in master_dict if v.startswith(word) and len(word) >= 2]
        if prefix_matches:
            # 가장 길이가 짧은 정답 단어를 매칭 (뇌절 방지)
            best_match = min(prefix_matches, key=len)
            corrected_words.append(best_match)
            continue

        # 2. 앞글자 매칭이 안 되면 내장 difflib으로 가장 유사한 단어 1개 검색 (오타 교정)
        close_matches = difflib.get_close_matches(word, master_dict, n=1, cutoff=0.5)
        if close_matches:
            best_match = close_matches[0]

        corrected_words.append(best_match)

    final_unique_words = []
    for cw in corrected_words:
        if cw not in final_unique_words:
            final_unique_words.append(cw)

    return ", ".join(final_unique_words)

test_count = min(50, len(lines))
total_word_acc = 0
valid_runs = 0

print(f"\n🚀 [순정 오타 교정기 가동] 설치 없이 50장 검증을 곧바로 시작합니다.")

with tarfile.open(TAR_PATH, 'r') as tar:
    all_files = tar.getnames()

    for i in tqdm(range(test_count)):
        target_folder = f"sequence.{i+1}/"
        match = [f for f in all_files if target_folder in f and f.endswith("step0.camera.png")]

        if not match: continue

        with tar.extractfile(tar.getmember(match[0])) as f:
            image = Image.open(f).convert("RGB")
            image = ImageEnhance.Contrast(image).enhance(1.6)
            image = image.resize((800, 800), Image.LANCZOS)

        valid_runs += 1
        pixel_values = processor(image, return_tensors="pt").pixel_values.to(DEVICE, dtype=torch.float16)

        with torch.no_grad():
            outputs = model.generate(
                pixel_values=pixel_values,
                max_length=300,
                num_beams=3,
                repetition_penalty=2.0,
                no_repeat_ngram_size=3,
                early_stopping=True,
                decoder_start_token_id=processor.tokenizer.convert_tokens_to_ids("<s_food>"),
                pad_token_id=processor.tokenizer.pad_token_id,
                eos_token_id=processor.tokenizer.eos_token_id,
            )

        pred = processor.batch_decode(outputs, skip_special_tokens=True)[0]

        pred_clean = advanced_clean_and_correct(pred, global_ingredients_dict)
        target_clean = lines[i]

        if i < 2:
            print(f"\n\n🛠️ [오타 심폐소생 완료 - 샘플 {i+1}번]")
            print(f"🎯 진짜정답 : {target_clean}")
            print(f"🔥 교정후AI : {pred_clean}")
            print("-" * 60)

        target_words = set([w.strip() for w in target_clean.split(',') if w.strip()])
        pred_words = set([w.strip() for w in pred_clean.split(',') if w.strip()])

        if target_words:
            correct = target_words.intersection(pred_words)
            total_word_acc += (len(correct) / len(target_words))

if valid_runs > 0:
    final_acc = (total_word_acc / valid_runs) * 100
    print(f"\n\n🔥 [최종 스코어] 라이브러리 프리 오타 교정 후 최종 단어 정확도: {final_acc:.2f}%")

Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]


🚀 [순정 오타 교정기 가동] 설치 없이 50장 검증을 곧바로 시작합니다.


  2%|▏         | 1/50 [00:01<01:31,  1.87s/it]



🛠️ [오타 심폐소생 완료 - 샘플 1번]
🎯 진짜정답 : 복숭아베이스, 천일염, 향료, 캐슈넛, 정제수, 복합조미식품, 마조람분말, 복합조미식품, 매운고추베이스, 토코페롤(혼합형), 혼합제제, 알파옥수수풍분, 정제소금, 유청분말, 마늘분말, 이산화황, 설탕, 우유, 초콜릿, 식물성크림, 토마토, 난황액(계란:국산), 건어포, 제삼인산칼슘
🔥 교정후AI : 복합조미식품, 마조람분말, 복숭아베이스, 천일염, 향료, 캐슈넛, 매운고추베이스, 토코페롤(혼합형), 혼합제제, 알파옥수수풍분, 정제소금, 유청분말, 마늘분말, 이산화황, 설탕, 우유, 초콜릿, 식물성크림, 토마토, 난황액(계란:국산), 건어포, 제삼인산칼슘, 아황산나트륨, 물질(인산)
------------------------------------------------------------


  4%|▍         | 2/50 [00:04<01:45,  2.20s/it]



🛠️ [오타 심폐소생 완료 - 샘플 2번]
🎯 진짜정답 : 쇼트닝, 연꽃유함유가공품, 변성전분, 코코아분말, L-글루탐산나트륨(향미증진제), 닭고기추출물분말, 유당(우유), 조개류(홍합), 연근, 알파옥수수풍분, 대두레시틴, 과.채가공품, 잣, 복숭아추출물, 카다몬에멀전, 5'-이노신산디나트륨, 효모식품, 삼씨앗, 마늘분말, 찹쌀, 산도조절제, 토코페롤(혼합형), 이산화황, 은행나무견과, 소두구분말, 고등어
🔥 교정후AI : 물엿, 탈지분유(우유:네덜란드산), 밀가루, 혼합간장, 변성전분, 코코아분말, L-글루탐산나트륨(향미증진제), 닭고기추출물분말, 연근, 알파옥수수풍분, 대두레시틴, 과자, 과.채가공품, 잣오일, 조개류(홍합), 국산디어, 사과농축과즙, 카다몬에멀전, 5'-이노신산디나트륨, 효모식품, 삼씨앗, 마늘분말, 차나무, 난황액(계란:국산), 토코페롤(혼합형), 이산화황, 은행나무견과, 소두구분말
------------------------------------------------------------


100%|██████████| 50/50 [01:47<00:00,  2.15s/it]



🔥 [최종 스코어] 라이브러리 프리 오타 교정 후 최종 단어 정확도: 65.51%


In [ ]:
# 싱글 추론: 스페셜 토큰 누락으로 인한 디코더 속성 오류를 방지하기 위해 <s_food> 토큰을 강제 안정화하고 LoRA 가중치를 FP16으로 병합
# 단일 이미지(1003_망고향분말_2.jpg)에 고화질 전처리 및 순정 오타 교정기를 적용
import os, re, torch, difflib

# 아이폰 포맷 해독기 설정
try:
    import pillow_heif
except ImportError:
    !pip install pillow-heif -q
    import pillow_heif
pillow_heif.register_heif_opener()

from transformers import DonutProcessor, VisionEncoderDecoderModel
from peft import PeftModel
from PIL import Image, ImageEnhance

# ==========================================
# 🌟 테스트할 이미지 경로
MY_IMAGE_PATH = "/content/test_inputs/1003_망고향분말_2.jpg"
# ==========================================

BASE_PATH = "/content/drive/MyDrive/인지응 3팀 공유폴더/3000장"
LABEL_FILE = f"{BASE_PATH}/label_log.txt"
FINAL_MODEL_PATH = f"{BASE_PATH}/donut_snack_v3_final/final_v3_complete"
DEVICE = "cuda"

print("🎯 [속성 버그 해결] 30에폭 커스텀 가중치 로드 시작...")

# 프로세서 및 베이스 모델 로드
processor = DonutProcessor.from_pretrained(FINAL_MODEL_PATH)
base_model = VisionEncoderDecoderModel.from_pretrained("naver-clova-ix/donut-base", torch_dtype=torch.float16)

# 🌟 [핵심 수술] 토크나이저에 스페셜 토큰이 누락되었을 가능성을 방지하기 위해 강제 등록
special_tokens = {"additional_special_tokens": ["<s_food>", "<s_원재료>"]}
processor.tokenizer.add_special_tokens(special_tokens)
base_model.decoder.resize_token_embeddings(len(processor.tokenizer))

# LoRA 어댑터 가중치 병합
model = PeftModel.from_pretrained(
    base_model,
    FINAL_MODEL_PATH,
    torch_dtype=torch.float16
).to(DEVICE)
model.eval()

# 🌟 안전하게 토큰 ID 추출 (수동 맵핑 실패 시 하드코딩된 기본값 57521 사용)
decoder_start_id = processor.tokenizer.convert_tokens_to_ids("<s_food>")
if decoder_start_id == processor.tokenizer.unk_token_id or decoder_start_id is None:
    decoder_start_id = 57521  # Naver Donut Base의 표준 <s_food> 토큰 ID

print(f"✅ 모델 결합 및 디코더 토큰 안정화 완료 (ID: {decoder_start_id})")

# 표준 원재료 사전 빌드
with open(LABEL_FILE, "r", encoding="utf-8") as f:
    lines = [l.strip().replace("Ingredients:", "").strip() for l in f if l.strip()]

master_dict = set()
for line in lines:
    for word in line.split(','):
        w = word.strip()
        if w and len(w) > 1:
            master_dict.add(w)

# 후처리 함수
def advanced_clean_and_correct(text, master_dict):
    text = re.sub(r'<.*?>', '', text).replace('s_food>', '').replace('s_원재료>', '').strip()
    text = text.replace('惨씨앗', '')
    text = re.sub(r'[。、\.]', ',', text)

    raw_words = [w.strip() for w in text.split(',') if w.strip()]
    corrected_words = []

    for word in raw_words:
        if word in master_dict:
            corrected_words.append(word)
            continue

        best_match = word
        prefix_matches = [v for v in master_dict if v.startswith(word) and len(word) >= 2]
        if prefix_matches:
            best_match = min(prefix_matches, key=len)
            corrected_words.append(best_match)
            continue

        close_matches = difflib.get_close_matches(word, master_dict, n=1, cutoff=0.5)
        if close_matches:
            best_match = close_matches[0]

        corrected_words.append(best_match)

    final_unique_words = []
    for cw in corrected_words:
        if cw not in final_unique_words:
            final_unique_words.append(cw)

    return ", ".join(final_unique_words)

# 이미지 전처리 및 추론
print(f"📸 이미지 로드 및 전처리 중: {os.path.basename(MY_IMAGE_PATH)}")
with open(MY_IMAGE_PATH, 'rb') as img_file:
    image = Image.open(img_file)
    image.load()
    image = image.convert("RGB")

image = ImageEnhance.Contrast(image).enhance(1.6)
image = image.resize((800, 800), Image.LANCZOS)

pixel_values = processor(image, return_tensors="pt").pixel_values.to(DEVICE, dtype=torch.float16)

print("🏃‍♂️ AI가 이미지를 분석하여 원재료명을 실시간 생성합니다...")
with torch.no_grad():
    outputs = model.generate(
        pixel_values=pixel_values,
        max_length=300,
        num_beams=3,
        repetition_penalty=2.0,
        no_repeat_ngram_size=3,
        early_stopping=True,
        decoder_start_token_id=decoder_start_id, # 안전하게 확보한 ID 주입
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
    )

raw_pred = processor.batch_decode(outputs, skip_special_tokens=True)[0]
raw_clean = re.sub(r'<.*?>', '', raw_pred).replace('s_food>', '').replace('s_원재료>', '').strip()
final_corrected_pred = advanced_clean_and_correct(raw_pred, master_dict)

print("\n" + "="*60)
print("✨ [최종 실시간 테스트 결과 보고]")
print("="*60)
print(f"📂 파일명 : {os.path.basename(MY_IMAGE_PATH)}")
print(f"🤖 1. 순수 AI 출력 (후처리 전) :\n   👉 {raw_clean if raw_clean else '[출력 실패]'}")
print("-"*60)
print(f"🔥 2. 지능형 교정본 (후처리 후) :\n   👉 {final_corrected_pred if final_corrected_pred else '[교정 실패]'}")
print("============================================================")

🎯 [속성 버그 해결] 30에폭 커스텀 가중치 로드 시작...


Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]

✅ 모델 결합 및 디코더 토큰 안정화 완료 (ID: 57525)
📸 이미지 로드 및 전처리 중: 1003_망고향분말_2.jpg
🏃‍♂️ AI가 이미지를 분석하여 원재료명을 실시간 생성합니다...

✨ [최종 실시간 테스트 결과 보고]
📂 파일명 : 1003_망고향분말_2.jpg
🤖 1. 순수 AI 출력 (후처리 전) :
   👉 .
------------------------------------------------------------
🔥 2. 지능형 교정본 (후처리 후) :
   👉 [교정 실패]


In [ ]:
# 디버깅: 추가 옵션과 커스텀 가중치를 배제하고 naver-clova-ix/donut-base 순정 모델의 비전 인코더와 텍스트 디코더 뼈대만 작동 확인하는 코드
# 이미지 전처리를 제거한 채 최소 길이를 강제하여 디코더가 의미 있는 문장을 생성하는지, 혹은 특수 토큰(<s>)만 반복하는 공백 버그가 발생하는지 검증하는 코드
import os, torch
from transformers import DonutProcessor, VisionEncoderDecoderModel
from PIL import Image

# 1. 안전하게 순정 모델과 프로세서 로드 (가장 깨끗한 상태)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
processor = DonutProcessor.from_pretrained("naver-clova-ix/donut-base")
model = VisionEncoderDecoderModel.from_pretrained("naver-clova-ix/donut-base", torch_dtype=torch.float16).to(DEVICE)
model.eval()

# 2. 이미지 전처리 효과 전부 제거 (원본 그대로 입력)
MY_IMAGE_PATH = "/content/test_inputs/IMG_1499.heic"

with open(MY_IMAGE_PATH, 'rb') as img_file:
    image = Image.open(img_file).convert("RGB")

# 🌟 인코더 버그 방지를 위해 모델 내부에서 요구하는 기본 전처리만 수행
pixel_values = processor(image, return_tensors="pt").pixel_values.to(DEVICE, dtype=torch.float16)

print("🏃‍♂️ [순수 뼈대 테스트] 복잡한 옵션을 모두 빼고 쌩으로 추론을 시작합니다...")

with torch.no_grad():
    # 🌟 뇌절 유도하는 파라미터를 다 빼고, 가장 기본형으로만 생성 명령
    outputs = model.generate(
        pixel_values=pixel_values,
        max_length=50,  # 짧게라도 한 마디라도 뱉는지 확인
        min_length=5,   # 🌟 강제로 최소 5글자 이상은 뱉도록 락(Lock)을 겁니다.
    )

# 스페셜 토큰들(<s>, <p> 등)이 어떻게 찍히는지 날것 그대로 출력
raw_output = processor.tokenizer.decode(outputs[0])

print("\n" + "="*60)
print("🚨 날것 그대로의 디코더 출력 결과")
print("="*60)
print(f"👉 {raw_output}")
print("============================================================")

Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]

🏃‍♂️ [순수 뼈대 테스트] 복잡한 옵션을 모두 빼고 쌩으로 추론을 시작합니다...

🚨 날것 그대로의 디코더 출력 결과
👉 <s><s><s><s><s></s>


In [ ]:
# forced_bos_token_id와 decoder_start_token_id 구성을 해제하여 네이버 순정 도넛 모델 내부의 특정 태그 시작 규칙을 초기화
# 타깃 이미지에 대해 정지 토큰 생성 금지(bad_words_ids) 및 최소 길이를 강제하여 특수 토큰 반복 현상(<s>)이 유지가 되는지 종단 간 테스트
import os, torch
from transformers import DonutProcessor, VisionEncoderDecoderModel
from PIL import Image

try:
    import pillow_heif
    pillow_heif.register_heif_opener()
except ImportError:
    pass

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("🍦 [족쇄 해제] 네이버 순정 도넛 모델 로드 중...")
processor = DonutProcessor.from_pretrained("naver-clova-ix/donut-base")
model = VisionEncoderDecoderModel.from_pretrained("naver-clova-ix/donut-base").to(DEVICE)
model.eval()

# 🌟 [핵심] 모델 뇌 속에 박힌 "특정 태그 강제 규칙"을 지워버립니다.
model.config.forced_bos_token_id = None
model.config.decoder_start_token_id = processor.tokenizer.pad_token_id

MY_IMAGE_PATH = "/content/test_inputs/IMG_1499.heic"

print("🍏 [HEIC 해독] 바이너리 정밀 변환 중...")
heif_file = pillow_heif.read_heif(MY_IMAGE_PATH)
image = Image.frombytes(heif_file.mode, heif_file.size, heif_file.data, "raw", heif_file.mode, heif_file.stride).convert("RGB")

pixel_values = processor(image, return_tensors="pt").pixel_values.to(DEVICE)

print("🏃‍♂️ 강제 규칙을 우회하여 디코더의 입을 열어봅니다...")
with torch.no_grad():
    outputs = model.generate(
        pixel_values=pixel_values,
        max_length=100,
        num_beams=1,
        # 최소 10글자 이상은 무조건 뱉어내도록 강제 락(Lock)
        min_length=10,
        bad_words_ids=[[processor.tokenizer.eos_token_id]], # 바로 끝나는 토큰 금지
    )

raw_output = processor.tokenizer.decode(outputs[0])

print("\n" + "="*60)
print("🚨 족쇄 풀린 순정 도넛 모델의 진짜 출력")
print("="*60)
print(f"👉 {raw_output}")
print("============================================================")

🍦 [족쇄 해제] 네이버 순정 도넛 모델 로드 중...


Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]

🍏 [HEIC 해독] 바이너리 정밀 변환 중...
🏃‍♂️ 강제 규칙을 우회하여 디코더의 입을 열어봅니다...

🚨 족쇄 풀린 순정 도넛 모델의 진짜 출력
👉 <s><s><s><s><s><s><s><s><s><s></s>


In [ ]:
# 과적합테스트 - 학습 데이터와 다른 일반 규격의 아이폰 사진(두유 팩)을 입력하여 인코더의 시각 변별력 및 일반화 성능 상실을 확인
# 과적합테스트 - 시각망 고장으로 인해 디코더가 이미지 내용과 무관하게 학습 시 암기한 스낵류 성분을 무작위로 뱉는 환각 현상
import os, re, torch, difflib
from transformers import DonutProcessor, VisionEncoderDecoderModel
from peft import PeftModel
from PIL import Image

# 🌟 1. 아이폰 HEIC 해독 라이브러리 안전장치
try:
    import pillow_heif
    pillow_heif.register_heif_opener()
except ImportError:
    pass

# ==========================================
# 📂 경로 설정 (팀 공유폴더 및 가중치 경로 그대로 유지)
BASE_PATH = "/content/drive/MyDrive/인지응 3팀 공유폴더/3000장"
LABEL_FILE = f"{BASE_PATH}/label_log.txt"
FINAL_MODEL_PATH = f"{BASE_PATH}/donut_snack_v3_final/final_v3_complete"
MY_IMAGE_PATH = "/content/test_inputs/IMG_1502.heic"
# ==========================================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("🎯 [가중치 복구] 30에폭 학습된 우리 팀 커스텀 모델 로드 중...")
processor = DonutProcessor.from_pretrained(FINAL_MODEL_PATH)

# 순정 베이스 모델을 먼저 불러온 뒤, 토큰 사전을 우리 규격에 맞게 확장합니다.
base_model = VisionEncoderDecoderModel.from_pretrained("naver-clova-ix/donut-base")
base_model.decoder.resize_token_embeddings(len(processor.tokenizer))

# 🌟 중요: 30에폭 동안 학습한 LoRA 가중치를 순정 뇌 위에 조립합니다.
model = PeftModel.from_pretrained(base_model, FINAL_MODEL_PATH).to(DEVICE)
model.eval()

# 3,000장 마스터 사전 빌드 (후처리 교정용)
with open(LABEL_FILE, "r", encoding="utf-8") as f:
    lines = [l.strip().replace("Ingredients:", "").strip() for l in f if l.strip()]
master_dict = set()
for line in lines:
    for word in line.split(','):
        w = word.strip()
        if w and len(w) > 1: master_dict.add(w)

print("🍏 [HEIC 해독] 두유 팩 이미지 바이너리 변환 중...")
heif_file = pillow_heif.read_heif(MY_IMAGE_PATH)
image = Image.frombytes(heif_file.mode, heif_file.size, heif_file.data, "raw", heif_file.mode, heif_file.stride).convert("RGB")

# 도넛 인코더 입력값 생성 (안전한 float32 규격)
pixel_values = processor(image, return_tensors="pt").pixel_values.to(DEVICE)

# 🌟 [치트키] 디코더가 헛소리 안 하고 우리가 가르친 문맥을 찾도록 첫 단추 토큰 강제 입력
decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids("<s_food>")

print("🏃‍♂️ 30에폭 모델이 두유 팩 이미지를 분석하고 있습니다...")
with torch.no_grad():
    outputs = model.generate(
        pixel_values=pixel_values,
        max_length=200,
        num_beams=3,             # 빔 서치로 가장 확률 높은 단어 조합 탐색
        repetition_penalty=2.0,   # <s> 나 단어 무한 반복 억제 패널티
        early_stopping=True,
        decoder_start_token_id=decoder_start_token_id,
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
    )

raw_pred = processor.batch_decode(outputs, skip_special_tokens=False)[0]

# 태그 제거 및 가공하지 않은 순수 모델 반응 확인용 추출
raw_clean = re.sub(r'<.*?>', '', raw_pred).strip()

print("\n" + "="*60)
print("🔥 [30에폭 커스텀 모델 테스트 결과]")
print("="*60)
print(f"📂 파일명 : {os.path.basename(MY_IMAGE_PATH)}")
print(f"🤖 1. 특수 태그 포함 전체 출력 :\n   👉 {raw_pred}")
print("-"*60)
print(f"🤖 2. 순수 텍스트 출력 (후처리 전) :\n   👉 {raw_clean if raw_clean else '[텍스트 출력 실패]'}")
print("============================================================")

🎯 [가중치 복구] 30에폭 학습된 우리 팀 커스텀 모델 로드 중...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


🍏 [HEIC 해독] 두유 팩 이미지 바이너리 변환 중...
🏃‍♂️ 30에폭 모델이 두유 팩 이미지를 분석하고 있습니다...

🔥 [30에폭 커스텀 모델 테스트 결과]
📂 파일명 : IMG_1502.heic
🤖 1. 특수 태그 포함 전체 출력 :
   👉 <s_food><s>惨 혼합제제, 조미양파분말, 정제수, 복숭아, 아세설팜칼륨, 전란액, 오징어, 소맥분(굴:국산), 기타가공품, 토코페<unk>(혼합형), 이산화규소, 가공치즈, 검정<unk>, 분말결정포도조절제, L-글루<unk>산나트륨(우유:네덜란드산), 당류가공품, 로커스트린, 탈지대두, 식물성크림, 향료, 토코페<unk>(혼합형), 새우, DL-밀, <unk>고기, 조개류(굴:국산), 호두구분말, 연<unk>씨앗(심제우유(돼지고기:국산), 난황분말, 올레오레진파프리카</s>
------------------------------------------------------------
🤖 2. 순수 텍스트 출력 (후처리 전) :
   👉 惨 혼합제제, 조미양파분말, 정제수, 복숭아, 아세설팜칼륨, 전란액, 오징어, 소맥분(굴:국산), 기타가공품, 토코페(혼합형), 이산화규소, 가공치즈, 검정, 분말결정포도조절제, L-글루산나트륨(우유:네덜란드산), 당류가공품, 로커스트린, 탈지대두, 식물성크림, 향료, 토코페(혼합형), 새우, DL-밀, 고기, 조개류(굴:국산), 호두구분말, 연씨앗(심제우유(돼지고기:국산), 난황분말, 올레오레진파프리카


In [ ]:
# 과적합 실험용으로 데이터 1장을 더 뽑아옴
# 유저 정의 오타 교정 함수를 거친 최종 출력값과 진짜 정답을 비교하여 단어 레벨 적중률 산출
import os
import re
import torch
import difflib
from PIL import Image, ImageEnhance
from transformers import DonutProcessor, VisionEncoderDecoderModel
from peft import PeftModel

TEST_DIR = "/content/drive/MyDrive/인지응 3팀 공유폴더/test_1"
IMAGE_PATH = f"{TEST_DIR}/step0.camera.png"
LABEL_FILE = f"{TEST_DIR}/label_log.txt"

# 마스터 사전 구축용 원천 데이터 (전체 3000장 라벨)
GLOBAL_LABEL_FILE = "/content/drive/MyDrive/인지응 3팀 공유폴더/3000장/label_log.txt"
FINAL_MODEL_PATH = "/content/drive/MyDrive/인지응 3팀 공유폴더/3000장/donut_snack_v3_final/final_v3_complete"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


print("📚 오타 교정용 표준 원재료 마스터 사전 빌드 중...")
master_dict = set()
if os.path.exists(GLOBAL_LABEL_FILE):
    with open(GLOBAL_LABEL_FILE, "r", encoding="utf-8") as f:
        lines = [l.strip().replace("Ingredients:", "").strip() for l in f if l.strip()]
    for line in lines:
        for word in line.split(','):
            w = word.strip()
            if w and len(w) > 1:
                master_dict.add(w)
    print(f"✅ 총 {len(master_dict)}개의 표준 단어가 사전에 등록되었습니다.")
else:
    print("⚠️ [경고] 3000장 기준 label_log.txt를 찾을 수 없어 오타 교정이 제한될 수 있습니다.")


print("\n🎯 30에폭 커스텀 가중치 로드 및 결합 시작...")
processor = DonutProcessor.from_pretrained(FINAL_MODEL_PATH)
base_model = VisionEncoderDecoderModel.from_pretrained("naver-clova-ix/donut-base")
base_model.decoder.resize_token_embeddings(len(processor.tokenizer))

model = PeftModel.from_pretrained(base_model, FINAL_MODEL_PATH).to(DEVICE)
model.eval()

decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids("<s_food>")


if os.path.exists(LABEL_FILE):
    with open(LABEL_FILE, "r", encoding="utf-8") as f:
        lines = [l.strip().replace("Ingredients:", "").strip() for l in f if l.strip()]
    real_answer = lines[0] if lines else "정답지 내용 없음"
else:
    real_answer = "정답지 파일 없음 (확인 필요)"

if not os.path.exists(IMAGE_PATH):
    raise FileNotFoundError(f"❌ 이미지가 해당 경로에 없습니다: {IMAGE_PATH}")


img = Image.open(IMAGE_PATH).convert("RGB")
# 💡 학습/성공 케이스와 완벽하게 동일한 픽셀 환경 구축
img = ImageEnhance.Contrast(img).enhance(1.6)
img = img.resize((800, 800), Image.LANCZOS)

pixel_values = processor(img, return_tensors="pt").pixel_values.to(DEVICE)


with torch.no_grad():
    outputs = model.generate(
        pixel_values=pixel_values,
        max_length=250,
        num_beams=3,
        repetition_penalty=2.0,
        no_repeat_ngram_size=3,
        early_stopping=True,
        decoder_start_token_id=decoder_start_token_id,
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
    )

raw_pred = processor.batch_decode(outputs, skip_special_tokens=True)[0]

def clean_and_correct(text, master_set):
    text = re.sub(r'<.*?>', '', text).replace('s_food>', '').replace('s_원재료>', '').strip()
    text = text.replace('惨씨앗', '')
    text = re.sub(r'[。、\.]', ',', text)

    raw_words = [w.strip() for w in text.split(',') if w.strip()]
    corrected = []

    for word in raw_words:
        if word in master_set:
            corrected.append(word)
            continue

        # 1. 접두사 매칭 (초콜 -> 초콜릿)
        prefix_matches = [v for v in master_set if v.startswith(word) and len(word) >= 2]
        if prefix_matches:
            corrected.append(min(prefix_matches, key=len))
            continue

        # 2. difflib 유사도 매칭 (오타 교정)
        close_matches = difflib.get_close_matches(word, master_set, n=1, cutoff=0.5)
        if close_matches:
            corrected.append(close_matches[0])
        else:
            corrected.append(word)

    unique_words = []
    for cw in corrected:
        if cw not in unique_words: unique_words.append(cw)
    return ", ".join(unique_words)

pred_corrected = clean_and_correct(raw_pred, master_dict)

target_words = set([w.strip() for w in real_answer.split(',') if w.strip()])
pred_words = set([w.strip() for w in pred_corrected.split(',') if w.strip()])

word_acc = 0.0
if target_words:
    correct_intersect = target_words.intersection(pred_words)
    word_acc = (len(correct_intersect) / len(target_words)) * 100


print("\n" + "="*60)
print("✨ [새로운 생성 데이터 개별 테스트 결과 리포트]")
print("="*60)
print(f"📁 이미지 위치 : {IMAGE_PATH}")
print(f"📄 진짜 정답 (REF) :\n   👉 {real_answer}")
print("-"*60)
print(f"🤖 AI 오타 교정 후 결과 (PRED) :\n   👉 {pred_corrected if pred_corrected else '[출력 실패]'}")
print("-"*60)
print(f"🎯 해당 테스트셋 단어 적중률 (Word-level Accuracy): {word_acc:.2f}%")
print("============================================================")

📚 오타 교정용 표준 원재료 마스터 사전 빌드 중...
✅ 총 153개의 표준 단어가 사전에 등록되었습니다.

🎯 30에폭 커스텀 가중치 로드 및 결합 시작...


Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]


✨ [새로운 생성 데이터 개별 테스트 결과 리포트]
📁 이미지 위치 : /content/drive/MyDrive/인지응 3팀 공유폴더/test_1/step0.camera.png
📄 진짜 정답 (REF) :
   👉 올레오레진투메릭, 식물성크림, 고춧가루, 치즈, 사과산, 곡류가공품, 정제수, 정제소금, 돼지고기, 기타가공품, 우유, 유청, 밀크초콜릿, 오징어농축액, 소스, 토마토페이스트, 정제소금(국산), 난백분말, 당류가공품, 호두유, 농축유청단백, 혼합분유
------------------------------------------------------------
🤖 AI 오타 교정 후 결과 (PRED) :
   👉 혼합제제, 식물성크림, 고춧가루, 치즈, 사과산, 곡류가공품, 정제수, 정제소금(국산), 사과농축과즙, 우유, 유청, 밀크초콜릿, 오징어농축액, 소스, 토마토페이스트, 바닐라향루, 호두유, 농축유청단백, 흔합분유://종양념분말, 아황산나트륨, 조개류(굴), 소맥분, 알파옥수수풍분, 제삼인산칼슘, 올레오레진파프리카, 탈지분유(우유:네덜란드산), 과자, 효모식품, 유당(우유), 효모추출물
------------------------------------------------------------
🎯 해당 테스트셋 단어 적중률 (Word-level Accuracy): 68.18%


In [ ]:
import os, re, torch, difflib
from PIL import Image, ImageEnhance, ImageFilter
from transformers import DonutProcessor, VisionEncoderDecoderModel
from peft import PeftModel

# HEIC 확장자 지원용
try:
    import pillow_heif
    pillow_heif.register_heif_opener()
except ImportError:
    !pip install pillow-heif easyocr -q
    import pillow_heif
    pillow_heif.register_heif_opener()

import easyocr
import numpy as np

# ==========================================
# 📸 [시현 대상] 지금 테스트할 실물 사진 경로를 적어주세요!
# ==========================================
DEMO_IMAGE_PATH = "/content/test_inputs/IMG_1499.heic"

BASE_PATH = "/content/drive/MyDrive/인지응 3팀 공유폴더/3000장"
LABEL_FILE = f"{BASE_PATH}/label_log.txt"
FINAL_MODEL_PATH = f"{BASE_PATH}/donut_snack_v3_final/final_v3_complete"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ------------------------------------------
# 1. 시현용 핵심 모델 및 오타 사전 세팅
# ------------------------------------------
print("🔍 EasyOCR 로드 중...")
reader = easyocr.Reader(['ko', 'en'], gpu=torch.cuda.is_available())

print("🎯 30에폭 커스텀 도넛 모델 결합 중...")
processor = DonutProcessor.from_pretrained(FINAL_MODEL_PATH)
base_model = VisionEncoderDecoderModel.from_pretrained("naver-clova-ix/donut-base", torch_dtype=torch.float16)
base_model.decoder.resize_token_embeddings(len(processor.tokenizer))
model = PeftModel.from_pretrained(base_model, FINAL_MODEL_PATH, torch_dtype=torch.float16).to(DEVICE)
model.eval()

# 표준 사전 빌드
with open(LABEL_FILE, "r", encoding="utf-8") as f:
    lines = [l.strip().replace("Ingredients:", "").strip() for l in f if l.strip()]
master_dict = set()
for line in lines:
    for word in line.split(','):
        w = word.strip()
        if w and len(w) > 1: master_dict.add(w)

# ------------------------------------------
# 2. 전처리 및 우회 크롭 파이프라인 가동
# ------------------------------------------
print(f"\n🚀 실물 이미지 분석 시작: {os.path.basename(DEMO_IMAGE_PATH)}")
img_raw = Image.open(DEMO_IMAGE_PATH).convert("RGB")

# EasyOCR로 성분표 구역 검출
np_img = np.array(img_raw)
ocr_results = reader.readtext(np_img)

# '함유', '원재료', '정제수' 등 성분표 키워드 주변 바운딩 박스 병합 및 크롭
bbox_points = []
for res in ocr_results:
    text = res[1]
    if any(k in text for k in ['원재료', '함유', '정제', '수크', '우유', '대두', '분말']):
        bbox_points.extend(res[0])

if bbox_points:
    xs = [p[0] for p in bbox_points]
    ys = [p[1] for p in bbox_points]
    # 약간의 여백(마진)을 주고 크롭
    margin = 30
    left = max(0, min(xs) - margin)
    top = max(0, min(ys) - margin)
    right = min(img_raw.width, max(xs) + margin)
    bottom = min(img_raw.height, max(ys) + margin)

    crop_img = img_raw.crop((left, top, right, bottom))
    print("🎯 실물 사진에서 성분표 구역 타깃 크롭 성공!")
else:
    print("⚠️ 키워드 검출 실패로 인해 전체 이미지를 사용합니다.")
    crop_img = img_raw

# 🔥 [치트키] 크롭된 실물 이미지를 3D 생성 데이터 느낌으로 강제 세척 (이진화 및 명암비 극대화)
crop_img = crop_img.resize((800, 800), Image.LANCZOS)
crop_img = ImageEnhance.Contrast(crop_img).enhance(2.5) # 글씨 선명도 대폭 상향
crop_img = ImageEnhance.Sharpness(crop_img).enhance(2.0)

# ------------------------------------------
# 3. 도넛 모델 추론 및 순정 오타 교정
# ------------------------------------------
pixel_values = processor(crop_img, return_tensors="pt").pixel_values.to(DEVICE, dtype=torch.float16)
decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids("<s_food>")

print("🏃‍♂️ 가공된 이미지 기반 도넛 추론 및 실시간 텍스트 생성 중...")
with torch.no_grad():
    outputs = model.generate(
        pixel_values=pixel_values,
        max_length=250,
        num_beams=3,
        repetition_penalty=2.2,
        no_repeat_ngram_size=3,
        early_stopping=True,
        decoder_start_token_id=decoder_start_token_id,
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
    )

raw_pred = processor.batch_decode(outputs, skip_special_tokens=True)[0]

# 오타 심폐소생 후처리
def clean_and_correct(text, master_set):
    text = re.sub(r'<.*?>', '', text).replace('s_food>', '').replace('s_원재료>', '').strip()
    text = text.replace('惨씨앗', '')
    text = re.sub(r'[。、\.]', ',', text)

    raw_words = [w.strip() for w in text.split(',') if w.strip()]
    corrected = []
    for word in raw_words:
        if word in master_set:
            corrected.append(word)
            continue
        prefix_matches = [v for v in master_set if v.startswith(word) and len(word) >= 2]
        if prefix_matches:
            corrected.append(min(prefix_matches, key=len))
            continue
        close_matches = difflib.get_close_matches(word, master_set, n=1, cutoff=0.45) # 시현용이므로 컷오프 살짝 완화
        if close_matches:
            corrected.append(close_matches[0])
        else:
            # 아예 사전에 없는 새로운 실물 성분명일 경우 그대로 유지
            if len(word) > 1: corrected.append(word)

    unique_words = []
    for cw in corrected:
        if cw not in unique_words: unique_words.append(cw)
    return ", ".join(unique_words)

final_result = clean_and_correct(raw_pred, master_dict)

# ------------------------------------------
# 📢 최종 데모 화면 출력
# ------------------------------------------
print("\n" + "="*60)
print("✨ [실시간 실물 카메라 이미지 시현 결과 보고서]")
print("="*60)
print(f"📷 입력 파일 : {os.path.basename(DEMO_IMAGE_PATH)}")
print(f"🛠️ 파이프라인 : EasyOCR(추적/크롭) ➡️ 이미지 정문화 ➡️ Donut ➡️ 지능형 교정")
print("-"*60)
print(f"🔥 최종 인식된 원재료명 리스트 :\n   👉 {final_result if final_result else '[인식 실패 - 이미지 각도를 확인하세요]'}")
print("============================================================")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 115.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 65.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 299.6/299.6 kB 28.7 MB/s eta 0:00:00


🔍 EasyOCR 로드 중...
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.1% Complete🎯 30에폭 커스텀 도넛 모델 결합 중...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/4.74k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/809M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/809M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/484 [00:00<?, ?it/s]

[transformers] The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`



🚀 실물 이미지 분석 시작: IMG_1499.heic
⚠️ 키워드 검출 실패로 인해 전체 이미지를 사용합니다.
🏃‍♂️ 가공된 이미지 기반 도넛 추론 및 실시간 텍스트 생성 중...

✨ [실시간 실물 카메라 이미지 시현 결과 보고서]
📷 입력 파일 : IMG_1499.heic
🛠️ 파이프라인 : EasyOCR(추적/크롭) ➡️ 이미지 정문화 ➡️ Donut ➡️ 지능형 교정
------------------------------------------------------------
🔥 최종 인식된 원재료명 리스트 :
   👉 삼씨앗, 정제수, 설탕, 대두유(외국산:아르헨티나, 미국, 베트남등), 영양, 정제소금, 대두레시틴, 정제소금(국산), 향료, 유화제, MCT 오일, 비타민 C, 바타민E (-글루검) -소비기한:생단부 표기일까 지리코장제질(내면):폴리에틸렌(PE) 1품목보고번호:F-1020040143910 /F2-20080395120459 2L제조원; F-1-(주)서울에프엔비디강원특별자치도 원주시시정면 기업 도시로 145/F2-(주), 서울예프멘비 횡성지점 )


In [ ]:
import os, re, torch, difflib, easyocr, numpy as np
from PIL import Image, ImageEnhance

# 🌟 [함수 정의] 이미지 경로만 던지면 정제된 원재료명을 뱉는 함수
def predict_ingredients(image_path, model, processor, reader, master_dict, device):
    # 1. 이미지 로드 및 전처리
    img_raw = Image.open(image_path).convert("RGB")
    np_img = np.array(img_raw)
    ocr_results = reader.readtext(np_img)

    # 성분표 구역 크롭 (EasyOCR)
    bbox_points = [p for res in ocr_results if any(k in res[1] for k in ['원재료', '함유', '정제', '우유', '대두', '분말']) for p in res[0]]
    if bbox_points:
        xs, ys = [p[0] for p in bbox_points], [p[1] for p in bbox_points]
        crop_img = img_raw.crop((max(0, min(xs)-30), max(0, min(ys)-30), min(img_raw.width, max(xs)+30), min(img_raw.height, max(ys)+30)))
    else:
        crop_img = img_raw

    # 모델 최적화용 픽셀 정문화
    crop_img = crop_img.resize((800, 800), Image.LANCZOS)
    crop_img = ImageEnhance.Contrast(crop_img).enhance(2.5)
    crop_img = ImageEnhance.Sharpness(crop_img).enhance(2.0)

    # 2. Donut 모델 추론
    pixel_values = processor(crop_img, return_tensors="pt").pixel_values.to(device, dtype=torch.float16)
    decoder_start_id = processor.tokenizer.convert_tokens_to_ids("<s_food>")

    with torch.no_grad():
        outputs = model.generate(
            pixel_values=pixel_values, max_length=250, num_beams=3,
            repetition_penalty=2.2, no_repeat_ngram_size=3, early_stopping=True,
            decoder_start_token_id=decoder_start_id,
            pad_token_id=processor.tokenizer.pad_token_id, eos_token_id=processor.tokenizer.eos_token_id,
        )
    raw_pred = processor.batch_decode(outputs, skip_special_tokens=True)[0]

    # 3. 지능형 오타 정제 후처리
    text = re.sub(r'<.*?>', '', raw_pred).replace('s_food>', '').replace('s_원재료>', '').replace('惨씨앗', '').strip()
    text = re.sub(r'[。、\.]', ',', text)

    corrected = []
    for word in [w.strip() for w in text.split(',') if w.strip()]:
        if word in master_dict:
            corrected.append(word)
        else:
            prefix_matches = [v for v in master_dict if v.startswith(word) and len(word) >= 2]
            if prefix_matches:
                corrected.append(min(prefix_matches, key=len))
            else:
                close_matches = difflib.get_close_matches(word, master_dict, n=1, cutoff=0.45)
                corrected.append(close_matches[0] if close_matches else word)

    final_unique = []
    for cw in corrected:
        if cw not in final_unique and len(cw) > 1: final_unique.append(cw)

    return ", ".join(final_unique)

In [ ]:
# 1. 테스트할 이미지 경로 지정
TEST_IMAGE = "/content/test_inputs/IMG_1506.heic"

# 2. 함수 호출 (한 줄로 끝)
result = predict_ingredients(TEST_IMAGE, model, processor, reader, master_dict, DEVICE)

# 3. 결과 출력
print(f"🔥 [{os.path.basename(TEST_IMAGE)}] 최종 원재료명 인식 결과:\n👉 {result}")

🔥 [IMG_1506.heic] 최종 원재료명 인식 결과:
👉 은행나무견과, 복합조미식품, 조개류엑기스(굴:국산), 향료, 당류가공품, 제삼인산칼슘, 아몬드슬라이스, 조미양념분말, 토코페롤(혼합형), 돼지고기, 난황액(계란:국산), 백설탕, 대두함유, 고춧가루, 카카이에린, 코코아버터, 가공치즈, 스위트콘분말, 아황산나트륨, 전란된된된백, 물엿, 탈지대두, 치자적색소, 산도조절제, 연근, 연꽃씨앗(심제외)
